<a href="https://colab.research.google.com/github/SrihariniPechimuthu/Mini_Project/blob/main/01_USGS_Earthquake_Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Import Libraries
import requests
import pandas as pd
import numpy as np
import re
from datetime import datetime
from sqlalchemy import create_engine


In [5]:
print("Colab setup successful ")

Colab setup successful 


In [7]:
# Define the USGS API Endpoint
USGS_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"

In [8]:
# Parameters for a 1-Month Test Pull (january 2024 as a sample)
params = {
    "format": "geojson",
    "starttime": "2024-01-01",
    "endtime": "2024-01-31",
    "minmagnitude": 1
}

In [9]:
#API Request
response = requests.get(USGS_URL, params=params)

print("Status Code:", response.status_code)

Status Code: 200


In [10]:
#Inspect the Response
data = response.json()

print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['type', 'metadata', 'features', 'bbox'])


In [11]:
#Check Number of Earthquake Events
len(data["features"])

8370

In [12]:
# Look at ONE Earthquake Record
data["features"][0]

{'type': 'Feature',
 'properties': {'mag': 1.97,
  'place': '11 km E of Lincoln, Montana',
  'time': 1706659042190,
  'updated': 1706660137610,
  'tz': None,
  'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/mb90039663',
  'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=mb90039663&format=geojson',
  'felt': None,
  'cdi': None,
  'mmi': None,
  'alert': None,
  'status': 'reviewed',
  'tsunami': 0,
  'sig': 60,
  'net': 'mb',
  'code': '90039663',
  'ids': ',mb90039663,',
  'sources': ',mb,',
  'types': ',origin,phase-data,',
  'nst': 15,
  'dmin': 0.08685,
  'rms': 0.12,
  'gap': 83,
  'magType': 'ml',
  'type': 'earthquake',
  'title': 'M 2.0 - 11 km E of Lincoln, Montana'},
 'geometry': {'type': 'Point',
  'coordinates': [-112.5305, 46.9431666666667, 2.86]},
 'id': 'mb90039663'}

In [13]:
# JSON → PANDAS DATAFRAME (26 FEATURES)
records = []

In [15]:
# Safely Extract Fields
for event in data["features"]:
    prop = event.get("properties", {})
    geo = event.get("geometry", {}).get("coordinates", [None, None, None])

    records.append({
        "id": event.get("id"),
        "time": pd.to_datetime(prop.get("time"), unit="ms"),
        "updated": pd.to_datetime(prop.get("updated"), unit="ms"),
        "longitude": geo[0],
        "latitude": geo[1],
        "depth_km": geo[2],
        "mag": prop.get("mag"),
        "magType": prop.get("magType"),
        "place": prop.get("place"),
        "status": prop.get("status"),
        "tsunami": prop.get("tsunami"),
        "sig": prop.get("sig"),
        "net": prop.get("net"),
        "nst": prop.get("nst"),
        "dmin": prop.get("dmin"),
        "rms": prop.get("rms"),
        "gap": prop.get("gap"),
        "magError": prop.get("magError"),
        "depthError": prop.get("depthError"),
        "magNst": prop.get("magNst"),
        "locationSource": prop.get("locationSource"),
        "magSource": prop.get("magSource"),
        "types": prop.get("types"),
        "ids": prop.get("ids"),
        "sources": prop.get("sources"),
        "type": prop.get("type")
    })

In [16]:
# Create the DataFrame
df = pd.DataFrame(records)

In [17]:
df.shape
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16740 entries, 0 to 16739
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id              16740 non-null  object        
 1   time            16740 non-null  datetime64[ns]
 2   updated         16740 non-null  datetime64[ns]
 3   longitude       16740 non-null  float64       
 4   latitude        16740 non-null  float64       
 5   depth_km        16740 non-null  float64       
 6   mag             16740 non-null  float64       
 7   magType         16740 non-null  object        
 8   place           16740 non-null  object        
 9   status          16740 non-null  object        
 10  tsunami         16740 non-null  int64         
 11  sig             16740 non-null  int64         
 12  net             16740 non-null  object        
 13  nst             10788 non-null  float64       
 14  dmin            9518 non-null   float64       
 15  rm

In [18]:
#Confirm Feature Count
len(df.columns)

26

In [19]:
# FETCH 5 YEARS OF DATA (MONTHLY, SAFE & SCALABLE)
# Create a Reusable Fetch Function
def fetch_monthly_earthquakes(start_date, end_date):
    params = {
        "format": "geojson",
        "starttime": start_date,
        "endtime": end_date,
        "minmagnitude": 1
    }

    try:
        response = requests.get(USGS_URL, params=params, timeout=30)
        response.raise_for_status()
        return response.json().get("features", [])
    except Exception as e:
        print(f"❌ Error fetching data for {start_date} to {end_date}: {e}")
        return []


In [20]:
# Define 5-Year Monthly Date Range
from dateutil.relativedelta import relativedelta

start_date = datetime(2019, 1, 1)
end_date = datetime(2024, 12, 31)


In [21]:
# Loop Month-by-Month
all_records = []

current = start_date

while current <= end_date:
    next_month = current + relativedelta(months=1)

    print(f"📥 Fetching data: {current.date()} to {next_month.date()}")

    monthly_data = fetch_monthly_earthquakes(
        current.strftime("%Y-%m-%d"),
        next_month.strftime("%Y-%m-%d")
    )

    for event in monthly_data:
        prop = event.get("properties", {})
        geo = event.get("geometry", {}).get("coordinates", [None, None, None])

        all_records.append({
            "id": event.get("id"),
            "time": pd.to_datetime(prop.get("time"), unit="ms"),
            "updated": pd.to_datetime(prop.get("updated"), unit="ms"),
            "longitude": geo[0],
            "latitude": geo[1],
            "depth_km": geo[2],
            "mag": prop.get("mag"),
            "magType": prop.get("magType"),
            "place": prop.get("place"),
            "status": prop.get("status"),
            "tsunami": prop.get("tsunami"),
            "sig": prop.get("sig"),
            "net": prop.get("net"),
            "nst": prop.get("nst"),
            "dmin": prop.get("dmin"),
            "rms": prop.get("rms"),
            "gap": prop.get("gap"),
            "magError": prop.get("magError"),
            "depthError": prop.get("depthError"),
            "magNst": prop.get("magNst"),
            "locationSource": prop.get("locationSource"),
            "magSource": prop.get("magSource"),
            "types": prop.get("types"),
            "ids": prop.get("ids"),
            "sources": prop.get("sources"),
            "type": prop.get("type")
        })

    current = next_month

📥 Fetching data: 2019-01-01 to 2019-02-01
📥 Fetching data: 2019-02-01 to 2019-03-01
📥 Fetching data: 2019-03-01 to 2019-04-01
📥 Fetching data: 2019-04-01 to 2019-05-01
📥 Fetching data: 2019-05-01 to 2019-06-01
📥 Fetching data: 2019-06-01 to 2019-07-01
📥 Fetching data: 2019-07-01 to 2019-08-01
❌ Error fetching data for 2019-07-01 to 2019-08-01: 400 Client Error: Bad Request for url: https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2019-07-01&endtime=2019-08-01&minmagnitude=1
📥 Fetching data: 2019-08-01 to 2019-09-01
📥 Fetching data: 2019-09-01 to 2019-10-01
📥 Fetching data: 2019-10-01 to 2019-11-01
📥 Fetching data: 2019-11-01 to 2019-12-01
📥 Fetching data: 2019-12-01 to 2020-01-01
📥 Fetching data: 2020-01-01 to 2020-02-01
📥 Fetching data: 2020-02-01 to 2020-03-01
📥 Fetching data: 2020-03-01 to 2020-04-01
📥 Fetching data: 2020-04-01 to 2020-05-01
📥 Fetching data: 2020-05-01 to 2020-06-01
📥 Fetching data: 2020-06-01 to 2020-07-01
📥 Fetching data: 2020-07-01 to 202

In [22]:
#Create the Full 5-Year DataFrame
df = pd.DataFrame(all_records)

In [23]:
#Validate the Dataset
df.shape

(666947, 26)

In [24]:
df["time"].min(), df["time"].max()

(Timestamp('2019-01-01 00:04:00.765000'),
 Timestamp('2024-12-31 23:51:43.276000'))

In [25]:
#DATA CLEANING, REGEX & FEATURE ENGINEERING
#Basic Dataset Health Check
df.isnull().sum().sort_values(ascending=False).head(10)

,0
locationSource,666947
magError,666947
depthError,666947
magNst,666947
magSource,666947
nst,279341
dmin,253822
gap,214391
rms,66
place,34


In [26]:
#Clean & Normalize Text Fields
Normalize text columns (lowercase + strip)
text_cols = [
    "magType", "status", "type", "net",
    "locationSource", "magSource"
]

for col in text_cols:
    df[col] = df[col].astype(str).str.lower().str.strip()

In [28]:
#REGEX – Extract Country from
def extract_country(place):
    if pd.isna(place):
        return "unknown"
    match = re.search(r",\s*([^,]+)$", place)
    return match.group(1).strip().lower() if match else "unknown"

df["country"] = df["place"].apply(extract_country)


In [29]:
df[["place", "country"]].sample(5)


,place,country
232294,"5 km SSW of Indios, Puerto Rico",puerto rico
551052,"60 km ENE of Pedro Bay, Alaska",alaska
446190,"27 km ESE of Clam Gulch, Alaska",alaska
434327,Reykjanes Ridge,unknown
232517,"4 km SSE of Maria Antonia, Puerto Rico",puerto rico


In [30]:
#Convert Numeric Fields Properly
numeric_cols = [
    "mag", "depth_km", "nst", "dmin", "rms",
    "gap", "magError", "depthError", "magNst", "sig"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")


In [31]:
#Handle Missing Numeric Values
df[numeric_cols] = df[numeric_cols].fillna(0)


In [32]:
#Time-Based Derived Columns
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day
df["day_of_week"] = df["time"].dt.day_name()
df["hour"] = df["time"].dt.hour


In [33]:
#Depth & Severity Classification
#Depth Category
df["depth_category"] = pd.cut(
    df["depth_km"],
    bins=[-1, 70, 300, 1000],
    labels=["shallow", "intermediate", "deep"]
)


In [34]:
#Strong / Destructive Earthquake Flag
df["strong_eq"] = (df["mag"] >= 7.5).astype(int)


In [35]:
#Final Validation
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 666947 entries, 0 to 666946
Data columns (total 34 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   id              666947 non-null  object        
 1   time            666947 non-null  datetime64[ns]
 2   updated         666947 non-null  datetime64[ns]
 3   longitude       666947 non-null  float64       
 4   latitude        666947 non-null  float64       
 5   depth_km        666947 non-null  float64       
 6   mag             666947 non-null  float64       
 7   magType         666947 non-null  object        
 8   place           666913 non-null  object        
 9   status          666947 non-null  object        
 10  tsunami         666947 non-null  int64         
 11  sig             666947 non-null  int64         
 12  net             666947 non-null  object        
 13  nst             666947 non-null  float64       
 14  dmin            666947 non-null  flo

In [36]:
df.describe()[["mag", "depth_km", "sig"]]


,mag,depth_km,sig
count,666947.000000,666947.000000,666947.000000
mean,2.118021,31.077529,88.616442
min,1.000000,-10.000000,15.000000
25%,1.300000,5.400000,26.000000
50%,1.740000,10.000000,47.000000
75%,2.430000,32.563001,92.000000
max,8.200000,681.238000,2910.000000
std,1.107595,63.212904,102.339971
